# Parsiranje scenarija: *Iskušenik* (*First Reformed*, 2017)

Izdvajanje iskaza Ernsta Tolera iz scenarija za film *Iskušenik*. Ovo je jedan od četiri parsera urađena za master rad *Božji usamljeni ljudi: računarska analiza jezika protagonista u scenarijima Pola Šrejdera* (Univerzitet Singidunum, 2026). Svaki scenario u korpusu ima svoj parser,  četiri scenarija razlikuju po načinu na koji su zapisani ime lika, parentetika, dijalog i didaskalije. Postupak je opisan u potpoglavljima 5.2 i 5.3 rada.

**Izvor.** Scenario je preuzet sa sajta scriptslug.com (tačna adresa navedena je u literaturi rada) kao PDF od 88 strana. Datiran je na 19. januar 2017, ima zaključanu numeraciju scena i ne sadrži naknadne revizije. Fajl nije deo repozitorijuma, jer je scenario zaštićen autorskim pravom. Za ponavljanje postupka PDF treba sačuvati kao 'First-Reformed-Script.pdf', u istom folderu kao notebook.

**Izlaz.** 'toller_lines.csv', sa kolonama 'speaker', 'type', 'record_no'i 'line': 248 iskaza, od toga 212 dijalog i 36 naracije. Ovaj fajl je, zajedno sa izlazima ostala tri parsera, ulaz za notebook 'Analiza_scenarija.ipynb'.

**Pokretanje.** Redom, od prve ćelije. Potrebni paketi: 'pdfplumber i 'pandas'.


In [1]:
import pdfplumber
import re
import pandas as pd
from collections import defaultdict
from collections import Counter

## 1. Učitavanje scenarija

PDF se čita strana po strana pomoću biblioteke pdfplumber, koja za svaku reč daje njen položaj na strani. Reči čija vertikalna pozicija, zaokružena na ceo poen, ima istu vrednost čine jedan red, a za svaki red beleže se strana, x-koordinata prve reči i tekst.

Pri tom se "pametni" apostrofi i navodnici prevode u obične. U ovom scenariju to nije samo ujednačavanje zapisa. Većina oznaka '(CONT'D)' napisana je sa "pametnim" apostrofom, pa je bez ove zamene obrazac za ime lika ne bi prepoznao, a replika ispod takve oznake bila bi tiho izgubljena.


In [2]:
PDF_PATH = 'First-Reformed-Script.pdf'

def normalizuj(t):
    # pretvaranje apostrofa i navodnika u obične; bez toga (CONT'D) sa "pametnim" apostrofom
    # ne bi bio prepoznat kao sufiks i replika ispod njega bi se izgubila
    return (t.replace('’', "'").replace('‘', "'")
            .replace('“', '"').replace('”', '"'))

pdf = pdfplumber.open(PDF_PATH)
all_rows = []
for pi, page in enumerate(pdf.pages):
    words = page.extract_words()
    rows = defaultdict(list)
    for w in words:
        rows[round(w['top'])].append(w) # grupisanje reči u isti red po y koordinati
    for y in sorted(rows):
        ws = sorted(rows[y], key=lambda w: w['x0'])
        x0 = round(ws[0]['x0']) # leva pozicija reda
        text = normalizuj(' '. join(w['text'] for w in ws))
        all_rows.append({'page': pi, 'x0': x0, 'text': text})

## 2. Prepoznavanje elemenata scenarija

Uvlačenje u ovom scenariju nije dosledno. Na desetak strana ime lika stoji na x = 240, 268 ili 281 umesto na 252, a dijalog na 168, 193 ili 311 umesto na 180. Jedno 'TOLLER' stoji čak na x = 179, u pojasu dijaloga. Zato fiksni opsezi x-koordinate, kakvi su korišćeni za *Fatalnu nesanicu*, ovde ne bi radili.

Položaj zato služi samo da odvoji krajeve strane: sve što počinje levo od x = 140 je didaskalija ili zaglavlje scene (x ≈ 108, odnosno 54), a sve desno od x = 440 je prelaz, kao 'CUT To'. U srednjem pojasu tip reda određuje njegov oblik. Red koji počinje zagradom je parentetika, red koji odgovara obrascu imena lika je ime, a sve ostalo je dijalog.

Ime lika je zapisano velikim slovima, može sadržati cifre, razmak, tačku, apostrof i crticu, i ne završava se tačkom. Poslednji uslov sprečava da se kratak red dijaloga pisan velikim slovima, kao 'PM.', prepozna kao ime. Uz ime mogu stajati sufiksi u zagradama, '(V.O.)', '(CONT'D)', '(O.S.)' i '(O.C.)', i to više njih zaredom, kao u 'TOLLER (V.O.) (CONT'D)'.

Na kraju ćelije obrazac se proverava na nekoliko primera, uključujući i 'PM.', koji ne sme da prođe kao ime.


In [3]:
speaker_name = re.compile(
    r"^([A-Z0-9][A-Z0-9 .'\-]{1,28}?)(?<!\.)"
    r"((?:\s*\((?:V\.O\.?|CONT'?D|O\.S\.?|O\.C\.?)\))*)\s*$"
)

# uvlačenje je nedosledno na desetak strana, pa pozicija odvaja samo didaskalije i prelaze, 
# a ime lika i parentetika se prepoznaju po obliku reda
def classify(x0, txt):
    if x0 <140: return 'action' # didaskalije (108) i zaglavlje scena (54)
    if x0 > 440: return 'other' # CUT TO itd
    if txt.startswith('('):return 'paren'
    if speaker_name.match(txt): return 'speaker'
    return 'dialogue'

# provera obrasca na primerima; PM. je red dijaloga koji ne sme da prođe kao ime
for t in ["TOLLER", "TOLLER (V.O.)", "TOLLER (V.O.) (CONT'D)", "MARY (O.S.)", "PM."]:
    print (t, bool(speaker_name.match(t)))

TOLLER True
TOLLER (V.O.) True
TOLLER (V.O.) (CONT'D) True
MARY (O.S.) True
PM. False


## 3. Parsiranje

Parser prolazi kroz redove redom i od njih sklapa iskaze. Ime lika otvara nov iskaz, redovi dijaloga dodaju se tekućem iskazu, a didaskalija ili prelaz ga zatvara. Pri zatvaranju se redovi spajaju u jedan tekst, a višestruki razmaci svode na jedan.

Ako uz ime stoji sufiks '(V.O.)', ceo iskaz se označava kao naracija. Uslov traži 'V.O' bez završne tačke, jer se u scenariju javlja i oblik 'TOLLER (V.O)'. Tolerovi dnevnički zapisi dosledno su označeni ovim sufiksom, pa za njih nije bilo potrebno posebno pravilo.

Parentetika ne ulazi u tekst iskaza i ne prekida ga, pa se delovi replike pre i posle nje spajaju u jedan iskaz. Kad parentetika prelazi u sledeći red, preskaču se svi redovi do onog u kojem se zagrada zatvara.

Replika prelomljena preko strane obeležena je oznakom '(MORE)' na dnu strane i ponovljenim imenom lika sa '(CONT'D)' na vrhu sledeće. U tom slučaju parser ne otvara nov iskaz, nego nastavlja tekući, pa prelom strane ne deli repliku na dva iskaza. Oznaka '(CONT'D)' bez prethodnog '(MORE)', na primer kad lik nastavlja da govori posle didaskalije, otvara nov iskaz, kao i svako drugo ime.


In [4]:
records = []
current, current_vo, buf = None, False, []
more, in_paren = False, False

def flush():
    'Zatvori tekuću repliku i upiši je'
    global buf
    if current and buf:
        txt = re.sub(r'\s+', ' ', ' '.join(buf)).strip()
        if txt:
            records.append((current, current_vo, txt))
    buf = []

for r in all_rows:
    txt = r['text'].strip()

    # nastavak parentetike koja prelazi u sledeći red
    if in_paren:
        if ')' in txt:
            in_paren = False
        continue

    cls = classify(r['x0'], txt)

    if cls == 'speaker':
        m = speaker_name.match(txt)
        ime, sufiksi = m.group(1).strip(), m.group(2)
        # replika prelomljena preko strane: (MORE), isto ime sa (CONT'D)
        if more and ime == current and 'CONT' in sufiksi:
            more = False
            continue
        flush()
        more = False
        current = ime
        current_vo = 'V.O' in sufiksi # bez završne tačke, jer se javlja i (V.O)
    elif cls == 'paren':
        if txt == '(MORE)':
            more = True
        elif ')' not in txt:
            in_paren = True
        continue
    elif cls == 'dialogue':
        if current:
            buf.append(txt)
    else:
        flush()
        current = None
        more = False
flush()

### Kontrola: oblici parentetike

Pregled parentetika u scenariju po učestalosti. U ispisu nema oznake naracije u obliku parentetike, jer se naracija u ovom scenariju označava sufiksom uz ime lika, a '(MORE)' se javlja 11 puta.

In [5]:
# kontrola: oblici parentetike i broja preloma (MORE)
paren =  [r['text'] for r in all_rows if classify(r['x0'], r['text']) == 'paren']
print(Counter(paren).most_common(30))

[('(beat)', 14), ('(MORE)', 11), ('(Toller nods)', 3), ('(nods)', 3), ('(she nods)', 3), ('(Michael nods)', 2), ('(grunt)', 2), ('(she sits)', 1), ('(he waits)', 1), ('(nods again)', 1), ('(about photos)', 1), ('(Toller waits)', 1), ('(Michael hesitates)', 1), ('(looking at teens)', 1), ('(to choir)', 1), ('(to Matt)', 1), ('(Toller shakes his head)', 1), ('(pantomimes playing organ)', 1), ('(shed lifts her eyebrows)', 1), ('(backs off)', 1), ('(looks away)', 1), ("(she doesn't understand)", 1), ('(shakes head)', 1), ('(reprimands)', 1), ('(gestures)', 1), ('(to owner)', 1), ('(to Toller and Jeffers)', 1), ('(they demur)', 1), ('(unfolds a sheet of paper)', 1), ('(Pause for effect)', 1)]


## 4. Izdvajanje Tolerovih iskaza i čuvanje

Svi iskazi se najpre smeštaju u tabelu i dobijaju redni broj u celom scenariju ('record_no'). Broj se dodeljuje pre izdvajanja protagoniste, pa obuhvata replike svih likova, dok didaskalije broj ne dobijaju. Zato dva Tolerova iskaza sa uzastopnim brojevima nije razdvojila replika drugog lika, nego najviše didaskalija. Na tome počiva spajanje iskaza naracije u blokove u glavnom notebooku.

Sve oznake koje sadrže ime 'TOLLER' svode se na jedno ime, pa se Toler vodi kao jedan govornik. Sufiksi poput '(V.O.)' i '(CONT'D)' ionako ne ulaze u ime.

U koloni 'type' vrednost 'dialogue' označava dijalog, a 'voice_over' ono što se u radu naziva naracijom. Očekivani rezultat je 248 iskaza, od toga 212 dijalog i 36 naracije.


In [6]:
df = pd.DataFrame(records, columns=['speaker_raw', 'is_vo', 'line'])
df['record_no'] = range(1, len(df) + 1) # redni broj u celom scenariju, potreban zbog kasnijeg spajanja blokova
# sve oznake koje sadrže TOLLER svode se na jedno ime
df['speaker'] = df['speaker_raw'].apply(lambda s: 'TOLLER' if 'TOLLER' in s else s)
toller = df[df['speaker'] == 'TOLLER'].copy().reset_index(drop=True)

toller['type'] = toller['is_vo'].apply(lambda v: 'voice_over' if v else 'dialogue')
toller = toller [['speaker', 'type', 'record_no', 'line']]

print('Tollerovih replika:', len(toller))
print(toller['type'].value_counts())

toller.to_csv('toller_lines.csv', index=False)


Tollerovih replika: 248
type
dialogue      212
voice_over     36
Name: count, dtype: int64
